<a href="https://colab.research.google.com/github/oskmaimloe-jp/rag_test_20261007/blob/main/chapter3/exercise1/exercise1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 演習の準備
---

## 必要なライブラリのインストール

In [1]:
# !pip install -q langchain langchain-core

In [2]:
!pip install -q langchain-openai==0.3.28 langchain-community==0.3.27

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.86 which is incompatible.
langgraph-sdk 0.4.5 requires langchain-core<2,>=1.4.0, but you have langchain-core

In [3]:
!pip freeze | grep langchain

langchain==0.3.30
langchain-community==0.3.27
langchain-core==0.3.86
langchain-openai==0.3.28
langchain-protocol==0.0.19
langchain-text-splitters==0.3.11


## API キーの設定
*  左ナビゲーションで [**シークレット**] アイコン (鍵形のアイコン) をクリックします。
*  [**新しいシークレットを追加**] をクリックし、[**名前**] に `OPENAI_API_KEY` と入力し、その [**値**] に指定されたキーを入力します。
*  [**新しいシークレットを追加**] をクリックし、[**名前**] に `LANGCHAIN_API_KEY` と入力し、その [**値**] に LangSmith で作成してコピーしておいた API キーを入力します。
*  設定した 2 つのシークレットの [**ノートブックからのアクセス**] を有効にします。
*  入力が完了したら、下のセルを実行します。

In [4]:
import os
from google.colab import userdata

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "default"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = userdata.get('LANGCHAIN_API_KEY')

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# 演習 1.1 会話履歴を保持しないチャットアプリ
---
各タスク (Task:) のコードを実装しながらセルを順次実行してください。

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

**Task:** Chat model のインスタンスを作成してください
* モデルのタイプ: OpenAI の `gpt-4o-mini`
* temperature: `0`

[langchain_openai.chat_models.base.ChatOpenAI](https://python.langchain.com/api_reference/openai/chat_models/langchain_openai.chat_models.base.ChatOpenAI.html)

In [6]:
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

**Task:** プロンプトのテンプレートを作成してください  
* SystemMessage と HumanMessage からテンプレートを作成します
* SystemMessage: 何か特定の分野についての専門性をもち、ユーザーの質問の回答するように指示してください (分野はお好きなもので構いません)
* HumanMessage: 後でユーザーの入力を代入できるように `{question}` という変数にしてください

[langchain_core.prompts.prompt.PromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.prompt.PromptTemplate.html)

In [7]:
chat_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "あなたは 生成AIの仕組みやRAG, APIに関して 高度なスキルをもつプロフェッショナルの研究者です。次の質問に答えてください："
    ),
    (
        "human",
        "{question}"
    )
])


**Task:** LLM の回答をテキストの形式に変換する Output parser のインスタンスを作成してください

[langchain_core.output_parsers.string.StrOutputParser](https://python.langchain.com/api_reference/core/output_parsers/langchain_core.output_parsers.string.StrOutputParser.html)

In [8]:
parser =StrOutputParser()

**Task:** `chat_prompt` > `model` > `parser` の順で処理を実行する Chain を定義してください

In [9]:
chain =chat_prompt | model | parser

次のセルを実行して動作を確認してください

In [10]:
print('Write Quit, Exit or Bye to quit.')
while True:
    q = input('Your prompt: ')
    if q.lower() in ['quit', 'exit', 'bye']:
        print('Quitting ... bye bye!')
        break

    answer = chain.invoke({"question": q})
    print(f'\nAnswer: {answer}')

Write Quit, Exit or Bye to quit.
Your prompt: こんにちは

Answer: こんにちは！どのようなことについてお話ししましょうか？
Your prompt: 現在の生成AIの進展に関して教えてください

Answer: 2023年10月時点での生成AIの進展には、いくつかの重要なトレンドと技術的な進化があります。以下に主なポイントを挙げます。

1. **大規模言語モデルの進化**: GPT-4やその後のモデルは、より多くのデータと計算リソースを使用してトレーニングされ、文脈理解や生成能力が向上しています。これにより、より自然で人間らしい対話が可能になっています。

2. **マルチモーダルAI**: テキストだけでなく、画像や音声など複数のデータ形式を扱うマルチモーダルモデルが進化しています。これにより、例えば画像を解析し、その内容に基づいてテキストを生成することが可能になっています。

3. **RAG（Retrieval-Augmented Generation）**: RAGは、生成モデルが外部の知識ベースやデータベースから情報を取得し、それを基に応答を生成する手法です。これにより、モデルはより正確で情報に基づいた回答を提供できるようになっています。

4. **APIの普及**: OpenAIやGoogleなどの企業が提供するAPIを通じて、生成AIの利用が広がっています。これにより、開発者は自分のアプリケーションにAI機能を簡単に統合できるようになり、さまざまな業界での応用が進んでいます。

5. **倫理と規制の議論**: 生成AIの進展に伴い、倫理的な問題や規制に関する議論も活発化しています。フェイクニュースの生成やプライバシーの侵害などのリスクに対処するためのガイドラインや法律が求められています。

6. **カスタマイズとファインチューニング**: ユーザーや企業が特定のニーズに応じてモデルをカスタマイズするための技術が進化しています。これにより、特定の業界や用途に特化した生成AIが実現されています。

これらの進展は、生成AIの能力を高めるだけでなく、さまざまな分野での応用を促進しています。今後も技術の進化とともに、新たな可能性が広がることが期待されます。
Your prompt: 可能性としてはど

LangSmish でトレース結果を参照し、想定した処理が実行されていたか確認してください。

# 演習 1.2 会話履歴を保持するチャットアプリ
---
下のセル内の各タスク (Task:) のコードを実装してください。  

[langchain_core.runnables.history.RunnableWithMessageHistory](https://python.langchain.com/api_reference/core/runnables/langchain_core.runnables.history.RunnableWithMessageHistory.html)  
[How to add memory to chatbots](https://python.langchain.com/docs/how_to/chatbots_memory/)

In [11]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate, AIMessagePromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_message_histories.in_memory import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Task:
# Chat model のインスタンスを作成
# ・OpenAI の gpt-4o-mini を使用

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)


# Task:
# 特定のセッション ID に対する会話履歴を取得する関数を作成
# ・ChatMessageHistory を使用

store = {}
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


# Task:
# Prompt template を作成
# ・SystemMessage と HumanMessage からテンプレートを作成します
# ・SystemMessage: 何か特定の分野についての専門性をもち、ユーザーの質問の回答するように指示します (分野はお好きなもので構いません)
# ・HumanMessage: 後でユーザーの入力を代入できるように {input} という変数にします
# ・MessgePlaceholder を使用してプロンプトに会話履歴を含めるようにします

chat_prompt =ChatPromptTemplate.from_messages(
  [
      SystemMessagePromptTemplate.from_template("あなたは生成AIの仕組みやRAG, APIに関して 高度なスキルをもつプロフェッショナルの研究者です。"),
      MessagesPlaceholder(variable_name="chat_history"),
      HumanMessagePromptTemplate.from_template("{input}")
  ]
)


# Task:
# Output parser を作成
# ・LLM からの出力をテキスト形式に変換する Output Parser のインスタンスを作成します

parser = StrOutputParser()

# Task:
# Chain を定義する

chain = chat_prompt | model | parser


# Task:
# RunnableWithMessageHistory を使用して、chain の処理に会話履歴が組み込まれるように実装してください

runnable_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)



/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


次のセルを実行して動作を確認してください

In [12]:
print('Write Quit, Exit or Bye to quit.')
while True:
    q = input('Your prompt: ')
    if q.lower() in ['quit', 'exit', 'bye']:
        print('Quitting ... bye bye!')
        break

    answer = runnable_with_history.invoke({"input": q}, config={"configurable": {"session_id": "b123"}})
    print(f'\nAnswer: {answer}')

Write Quit, Exit or Bye to quit.
Your prompt: こんにちは

Answer: こんにちは！どのようなことについてお話ししましょうか？生成AIやRAG、APIに関する質問があればお答えします。
Your prompt: 現在の生成AIの進展に関して教えてください

Answer: 生成AIは近年急速に進展しており、さまざまな分野での応用が進んでいます。以下にいくつかの主要な進展を挙げます。

1. **モデルの大規模化**: GPT-3やGPT-4のような大規模言語モデルは、数十億から数百億のパラメータを持ち、より自然な言語生成が可能になっています。これにより、文脈理解や創造性が向上しました。

2. **マルチモーダルAI**: テキストだけでなく、画像や音声など複数のデータ形式を扱うマルチモーダルモデルが登場しています。例えば、OpenAIのDALL-EやCLIPは、テキストから画像を生成したり、画像を理解したりする能力を持っています。

3. **ファインチューニングと適応**: 特定のタスクやドメインに特化したファインチューニングが進んでおり、特定の業界やニーズに応じたカスタマイズが容易になっています。これにより、生成AIはより実用的なツールとして利用されています。

4. **RAG（Retrieval-Augmented Generation）**: RAGは、生成モデルと情報検索を組み合わせたアプローチで、外部の知識ベースから情報を引き出し、それを基に生成を行うことができます。これにより、より正確で情報に基づいた応答が可能になります。

5. **倫理とバイアスの問題**: 生成AIの進展に伴い、倫理的な問題やバイアスの問題も注目されています。研究者たちは、AIの公平性や透明性を確保するための方法を模索しています。

6. **APIの普及**: OpenAIやGoogleなどの企業が提供するAPIを通じて、開発者は生成AIを簡単に利用できるようになっています。これにより、さまざまなアプリケーションやサービスに生成AIが組み込まれるようになっています。

これらの進展は、生成AIの能力を大幅に向上させ、さまざまな産業での利用を促進しています。今後もこの分野は進化し続けるでしょう。興味のある特定のトピックについて

LangSmish でトレース結果を参照し、想定した処理が実行されていたか確認してください。

お時間に余裕のある方は、プロンプトをいろいろ変えて試してみてください。  
コードを変更していただいても構いません。